In [2]:
!pip install PyTDC

     -------------------------------------- 154.2/154.2 KB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 315.1/315.1 KB 6.5 MB/s eta 0:00:00
     -------------------------------------- 542.1/542.1 KB 6.8 MB/s eta 0:00:00
     ---------------------------------------- 84.1/84.1 KB 2.3 MB/s eta 0:00:00
     -------------------------------------- 566.4/566.4 KB 5.9 MB/s eta 0:00:00
     -------------------------------------- 151.3/151.3 KB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 151.2/151.2 KB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 151.2/151.2 KB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing m

You should consider upgrading via the 'C:\Users\Veronika\anaconda3\python.exe -m pip install --upgrade pip' command.



     -------------------------------------- 151.1/151.1 KB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 151.1/151.1 KB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 151.2/151.2 KB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 151.2/151.2 KB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ---------------------------------------- 151.3/151.3 KB ? eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ---------------------------------------- 2.7/2.7 MB 6.9 MB/s eta 0:00:00
     -----------

In [22]:
from tdc.utils import retrieve_label_name_list
from tdc.single_pred import Tox

import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

import re

In [6]:
label_list = retrieve_label_name_list('Tox21')
label_list

['NR-AR',
 'NR-AR-LBD',
 'NR-AhR',
 'NR-Aromatase',
 'NR-ER',
 'NR-ER-LBD',
 'NR-PPAR-gamma',
 'SR-ARE',
 'SR-ATAD5',
 'SR-HSE',
 'SR-MMP',
 'SR-p53']

## Data loading
Stress-response p53 was selected, as I have an idea what it is responsible for: activation pf the p53 signaling stress-response pathway (carcinogenic). 

In [7]:
data = Tox(name = 'Tox21', label_name = 'SR-p53')
# split = data.get_split()
data

Downloading...
100%|██████████████████████████████████████████████████████████████████████████████| 712k/712k [00:00<00:00, 1.64MiB/s]
Loading...
Done!


## Data split
I used scaffold split instead of random split as it's claimed to ensure better training. 
Note: shall I test in the future the comparison of performance of models on random splitted data vs scaffold splitted data?

Description: Scaffold split is based on the scaffold of the molecules so that train/val/test set is more structurally different. It is more challenging than random split.

Note: Scaffold split only applies to single-instance drugs-related tasks (ADME, Tox, HTS). Scaffold split also requires RDKit installed.

Sources:
https://tdcommons.ai/functions/data_split/
https://pmc.ncbi.nlm.nih.gov/articles/PMC8627024/

In [17]:
split = data.get_split(method = 'scaffold')
train = split["train"]
valid = split["valid"]
test = split["test"]
train

,Drug_ID,Drug,Y
0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O,0.0
1,TOX5110,CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C,0.0
2,TOX22514,OC[C@H](O)[C@@H](O)[C@H](O)CO,0.0
3,TOX6612,CC(C)COC(=O)C(C)C,0.0
4,TOX6615,C=C(C)C(=O)OCCOC(=O)C(=C)C,0.0
...,...,...,...
4736,TOX7585,c1cc(N(CC2CO2)CC2CO2)ccc1OCC1CO1,1.0
4737,TOX26744,CC(C)(Cc1c[nH]c2ccccc12)NCC(O)COc1ccccc1C#N,0.0
4738,TOX923,Cc1cccc(C)c1NC(=O)CN1CCCC1=O,0.0
4739,TOX25680,CN(C(=O)c1c(O)c2ccccc2n(C)c1=O)c1ccccc1,0.0


### Split sizes and class balance
From the target values display I see, in the train set 0 is about 20 times more likely than 1, in the validation set 12 times more likely, and in the test set 8.5 more likely. So the class imbalance must be adressed on later steps.

In [18]:
for name, df in split.items():
    print(name)
    print(df.shape)
    print(df["Y"].value_counts(dropna=False))
    print()

train
(4741, 3)
0.0    4512
1.0     229
Name: Y, dtype: int64

valid
(677, 3)
0.0    625
1.0     52
Name: Y, dtype: int64

test
(1356, 3)
0.0    1214
1.0     142
Name: Y, dtype: int64



### Convert SMILES to RDKit fingerprints
Sources: https://www.rdkit.org/docs/source/rdkit.Chem.rdmolfiles.html



In [21]:
df = data.get_data()
isotope_pattern = re.compile(r"\[[0-9]+[A-Za-z]")

df["has_isotope_notation"] = df["Drug"].str.contains(isotope_pattern, regex=True)

df["has_isotope_notation"].value_counts()

False    6773
True        1
Name: has_isotope_notation, dtype: int64